In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
        .builder
        .appName("spark-local-app")
        .master("local[*]")
        .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/14 18:05:27 WARN Utils: Your hostname, DESKTOP-440A2SS, resolves to a loopback address: 127.0.1.1; using 172.20.131.26 instead (on interface eth0)
26/10/14 18:05:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/tem/de_lrn_prcts/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/14 18:05:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
datetimes = [
    (20261212, "12-Dec-2026 18:00:00.214"),
    (20261011, "11-Oct-2026 18:12:00.323"),
    (20260903, "03-Sep-2026 21:00:01.232")
]

In [3]:
df_datetimes = spark.createDataFrame(datetimes,
                                     schema="date BIGINT, time STRING")

In [5]:
df_datetimes.show(truncate=False)

+--------+------------------------+
|date    |time                    |
+--------+------------------------+
|20261212|12-Dec-2026 18:00:00.214|
|20261011|11-Oct-2026 18:12:00.323|
|20260903|03-Sep-2026 21:00:01.232|
+--------+------------------------+



In [12]:
from pyspark.sql.functions import col, to_date, to_timestamp

(
    df_datetimes
        .withColumn("to_date", to_date(col("date").cast("string"), "yyyyMMdd"))
        .withColumn("to_timestamp", to_timestamp(col("time"), "dd-MMM-yyyy HH:mm:ss.SSS"))
        .show(truncate=False)
)

+--------+------------------------+----------+-----------------------+
|date    |time                    |to_date   |to_timestamp           |
+--------+------------------------+----------+-----------------------+
|20261212|12-Dec-2026 18:00:00.214|2026-12-12|2026-12-12 18:00:00.214|
|20261011|11-Oct-2026 18:12:00.323|2026-10-11|2026-10-11 18:12:00.323|
|20260903|03-Sep-2026 21:00:01.232|2026-09-03|2026-09-03 21:00:01.232|
+--------+------------------------+----------+-----------------------+



In [14]:
from pyspark.sql.functions import date_format

(
    df_datetimes
        .withColumn("day_name", date_format(to_date(col("date").cast("string"), "yyyyMMdd"), "EE"))
        .show()
)

+--------+--------------------+--------+
|    date|                time|day_name|
+--------+--------------------+--------+
|20261212|12-Dec-2026 18:00...|     Sat|
|20261011|11-Oct-2026 18:12...|     Sun|
|20260903|03-Sep-2026 21:00...|     Thu|
+--------+--------------------+--------+



In [15]:
(
    df_datetimes
        .withColumn("day_name", date_format(to_date(col("date").cast("string"), "yyyyMMdd"), "EEEE"))
        .show()
)

+--------+--------------------+--------+
|    date|                time|day_name|
+--------+--------------------+--------+
|20261212|12-Dec-2026 18:00...|Saturday|
|20261011|11-Oct-2026 18:12...|  Sunday|
|20260903|03-Sep-2026 21:00...|Thursday|
+--------+--------------------+--------+



In [16]:
unixtimes = [(8947358493,),
             (4934093431,),
             (2382370452,),
             (2307121242,)
             ]

In [18]:
df_unixtimes = spark.createDataFrame(unixtimes).toDF("unixtime")

In [19]:
df_unixtimes.show()

+----------+
|  unixtime|
+----------+
|8947358493|
|4934093431|
|2382370452|
|2307121242|
+----------+



In [20]:
df_unixtimes.printSchema()

root
 |-- unixtime: long (nullable = true)



In [21]:
from pyspark.sql.functions import from_unixtime

df_unixtimes. \
    withColumn("date", from_unixtime("unixtime", "yyyy-MM-dd")). \
    withColumn("time", from_unixtime("unixtime")). \
    show()

+----------+----------+-------------------+
|  unixtime|      date|               time|
+----------+----------+-------------------+
|8947358493|2253-07-13|2253-07-13 12:21:33|
|4934093431|2126-05-10|2126-05-10 16:30:31|
|2382370452|2045-06-29|2045-06-29 20:34:12|
|2307121242|2043-02-09|2043-02-09 22:00:42|
+----------+----------+-------------------+



In [22]:
spark.stop()